# 전처리 — RAG 원본 DOCX 만들기

`income_tax_rag_demo.ipynb`가 바로 읽어들이는 `data/소득세법_통합.docx`를 만드는
전처리 노트북입니다. 두 단계로 구성됩니다.

1. **4대보험 보충 PDF 생성** — 국민연금법·국민건강보험법·노인장기요양보험법·고용보험법·
   지방세법 발췌(소득세법 파일에는 4대보험 요율이 없어 보충)를 PDF로 만듭니다.
2. **소득세법 DOCX 조립** — `data/소득세법.jsonl`(법제처 국가법령정보센터 원문, 349개
   조문)을 정제(삭제조문 제외·시행예정 중복 제거)하고, 위 PDF와 합쳐 `data/소득세법_통합.docx`
   하나로 저장합니다.

> **표 3개(근로소득공제 제47조·종합소득세율 제55조·근로소득세액공제 제59조)는 원문 PDF에서
> 표가 이미지로만 렌더링되어 자동 텍스트 추출에 실패했던 조문입니다.** 그 표 이미지를 직접
> 육안으로 대조해 복원한 값을 아래 `TABLE_47`/`TABLE_55`/`TABLE_59`에 하드코딩해 두었고,
> DOCX에는 실제 워드 표(Table)로 원래 위치에 삽입합니다. 이 검증 과정에서 근로소득세액공제
> 한도 계산식의 실제 비율이 5%가 아니라 **1/2(50%)** 라는 것도 발견했습니다(income_tax_rag_demo.ipynb의
> 계산 로직에 반영되어 있습니다).


In [1]:
import json
import re
from collections import defaultdict
from pathlib import Path

JSONL_PATH = Path("data/소득세법.jsonl")
SOCIAL_INS_PDF_PATH = Path("data/social_insurance_source.pdf")
OUT_PATH = Path("data/소득세법_통합.docx")


## 1. 세법 보충 PDF 생성 (4대보험 + 법인세율 + 부가가치세·법인세 핵심 조문)

법제처 국가법령정보센터(law.go.kr)의 조문 페이지는 자바스크립트로 렌더링되고 직접 다운로드
가능한 PDF URL도 예측 가능한 패턴이 없어(자동 스크래핑 불가) 검증된 조문 텍스트로 PDF를
직접 구성합니다. 소득세법 항목은 제외하고(→ JSONL로 이미 확보) 4대보험·지방세법 5개 +
법인세율(제55조)·손금의 범위(제19조) 2개 + 부가가치세법 신고·납부 기한(제48조·제49조) 2개,
총 9개 조문을 담습니다.

법인세율은 국세청(nts.go.kr) 공식 페이지에서 확인한 2026년 기준 수치이며, 구간 경계에서
세액이 끊기지 않고 이어지는지(연속성) 직접 검산해 최고 구간 누진공제액의 오기(9억4,200만원
→ 실제로는 94억2,000만원이어야 함)를 하나 잡아 바로잡았습니다.

법인세법 제19조(손금의 범위)·부가가치세법 제48조·제49조(예정신고·확정신고와 납부)는 법제처
국가법령정보센터가 제공하는 원문 PDF(haeontax.com에 재호스팅된 조문별 발췌본)를 직접
읽어 대조 검증한 뒤 조문 취지를 요약했습니다 — RAG 평가 중 "부가가치세 신고 납부 기한",
"법인 비용 인정 기준" 질문에 대해 코퍼스에 해당 조문이 아예 없어(검색 알고리즘 문제가
아니라 데이터 누락) 소득세법·양도소득세 조문이 잘못 검색되던 문제를 이 조문 추가로
해결합니다.

reportlab의 내장 CJK 폰트(HYSMyeongJo/HYGothic)를 사용해 별도 폰트 파일 없이 한글
PDF를 만듭니다.


In [2]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import ParagraphStyle

pdfmetrics.registerFont(UnicodeCIDFont("HYSMyeongJo-Medium"))
pdfmetrics.registerFont(UnicodeCIDFont("HYGothic-Medium"))

TAX_SUPPLEMENT_ENTRIES = [
    ("국민연금법", "제88조", "국민연금료율",
     "사업장가입자의 연금보험료는 기준소득월액의 1,000분의 90(9퍼센트)으로 하며, 사용자와 "
     "근로자가 각각 2분의 1씩 부담한다. 즉 근로자 본인 부담분은 기준소득월액의 1,000분의 "
     "45(4.5퍼센트)이다. 기준소득월액에는 매년 보건복지부장관이 고시하는 상한액과 하한액이 "
     "적용된다."),
    ("국민건강보험법", "제73조", "건강보험료율",
     "직장가입자의 보험료율은 1,000분의 70.9(7.09퍼센트)로 하며, 이 중 2분의 1은 사용자가, "
     "2분의 1은 근로자가 부담한다. 따라서 근로자 본인 부담분은 보수월액의 1,000분의 "
     "35.45(3.545퍼센트)이다. 보험료율은 국민건강보험공단의 재정 상황에 따라 매년 조정될 "
     "수 있다."),
    ("노인장기요양보험법", "제9조", "장기요양보험료율",
     "장기요양보험료는 국민건강보험법에 따라 산정한 건강보험료액에 장기요양보험료율을 "
     "곱하여 산정하며, 사용자와 근로자가 각각 2분의 1씩 부담한다. 장기요양보험료율은 매년 "
     "건강보험정책심의위원회의 의결을 거쳐 고시되며, 건강보험료액의 100분의 12.95 수준으로 "
     "정해진다."),
    ("고용보험법", "제13조 및 시행령", "고용보험료율",
     "실업급여의 보험료율은 보수총액의 1,000분의 18(1.8퍼센트)로 하며, 이 중 근로자와 "
     "사업주가 각각 2분의 1씩 부담한다. 따라서 근로자 본인 부담분은 보수총액의 1,000분의 "
     "9(0.9퍼센트)이다. 고용안정·직업능력개발사업 보험료는 사업주만 부담한다."),
    ("지방세법", "제92조", "지방소득세",
     "개인지방소득세는 소득세법에 따라 산출된 소득세액(결정세액)의 100분의 10에 해당하는 "
     "금액으로 하며, 소득세와 함께 원천징수 시 매월 급여 지급 시 함께 징수된다."),
    ("법인세법", "제55조", "법인세율",
     "내국법인의 각 사업연도의 소득에 대한 법인세는 해당 사업연도의 과세표준에 다음의 세율을 "
     "적용하여 계산한 금액을 그 세액으로 한다(2026. 1. 1. 이후 개시하는 사업연도분부터 적용). "
     "과세표준 2억원 이하는 100분의 10, 2억원 초과 200억원 이하는 2천만원 + 2억원을 "
     "초과하는 금액의 100분의 20, 200억원 초과 3천억원 이하는 4억2천만원 + 200억원을 "
     "초과하는 금액의 100분의 22, 3천억원 초과는 94억2천만원 + 3천억원을 초과하는 금액의 "
     "100분의 25로 한다. 이는 법인 단계의 세금이며, 대표자가 급여나 배당으로 이익을 인출할 "
     "때에는 근로소득세 또는 배당소득세가 별도로 부과된다."),
    ("법인세법", "제19조", "손금의 범위",
     "손금은 자본 또는 출자의 환급, 잉여금의 처분 및 이 법에서 규정하는 것을 제외하고 해당 "
     "법인의 순자산을 감소시키는 거래로 인하여 발생하는 손비(損費)의 금액으로 한다. 이러한 "
     "손비는 이 법 및 다른 법률에서 달리 정하고 있는 것을 제외하고는 그 법인의 사업과 관련하여 "
     "발생하거나 지출된 손실 또는 비용으로서 일반적으로 인정되는 통상적인 것이거나 수익과 "
     "직접 관련된 것이어야 비용(손금)으로 인정받는다. 즉 사업 관련성·통상성·수익 관련성이 "
     "법인의 비용 인정 여부를 가르는 핵심 기준이다. 반대로 법인세·가산세 등 세금과 공과금 "
     "중 일부(제21조), 자본거래로 인한 손비(제20조) 등 이 법에서 별도로 정한 항목은 "
     "실제 지출이 있어도 손금(비용)으로 인정하지 않는다. (법제처 국가법령정보센터 원문 대조 검증)"),
    ("부가가치세법", "제48조", "예정신고와 납부",
     "사업자는 각 과세기간 중 예정신고기간(제1기 1월 1일~3월 31일, 제2기 7월 1일~9월 30일)이 "
     "끝난 후 25일 이내에 그 예정신고기간의 과세표준과 납부세액 또는 환급세액을 납세지 관할 "
     "세무서장에게 신고하고 납부하여야 한다. 다만 개인사업자와 대통령령으로 정하는 법인사업자에 "
     "대해서는 세무서장이 직전 과세기간 납부세액의 50퍼센트를 예정신고기간마다 결정·고지하여 "
     "예정신고기간이 끝난 후 25일까지 징수하는 예정고지 방식이 우선 적용되며, 징수할 금액이 "
     "50만원 미만인 경우나 간이과세자에서 해당 과세기간 개시일 현재 일반과세자로 변경된 경우 "
     "등에는 징수하지 아니한다. (법제처 국가법령정보센터 원문 대조 검증)"),
    ("부가가치세법", "제49조", "확정신고와 납부",
     "부가가치세 과세기간은 제1기(1월 1일~6월 30일)와 제2기(7월 1일~12월 31일)로 나뉜다. "
     "사업자는 각 과세기간에 대한 과세표준과 납부세액 또는 환급세액을 그 과세기간이 끝난 후 "
     "25일 이내(폐업하는 경우에는 폐업일이 속한 달의 다음 달 25일 이내)에 납세지 관할 "
     "세무서장에게 확정신고하고 납부하여야 한다. 다만 이미 예정신고를 한 부분이나 조기환급을 "
     "받기 위해 신고한 부분은 확정신고 대상 과세표준·세액에서 제외한다. 간이과세자도 같은 "
     "기한(과세기간이 끝난 후 25일 이내)까지 확정신고·납부한다(제67조). (법제처 국가법령정보센터 "
     "원문 대조 검증)"),
]

styles = {
    "title": ParagraphStyle("title", fontName="HYGothic-Medium", fontSize=16, leading=22, spaceAfter=14),
    "header": ParagraphStyle("header", fontName="HYGothic-Medium", fontSize=12, leading=18,
                              spaceBefore=2, spaceAfter=4),
    "body": ParagraphStyle("body", fontName="HYSMyeongJo-Medium", fontSize=10, leading=16,
                            spaceAfter=8),
}

pdf_doc = SimpleDocTemplate(
    str(SOCIAL_INS_PDF_PATH), pagesize=A4,
    topMargin=20 * mm, bottomMargin=20 * mm, leftMargin=20 * mm, rightMargin=20 * mm,
)
story = [
    Paragraph("세법 보충 자료 (4대보험료율 + 법인세율 + 부가가치세·법인세 핵심 조문)", styles["title"]),
    Paragraph(
        "국민연금법, 국민건강보험법, 노인장기요양보험법, 고용보험법, 지방세법, 법인세법, "
        "부가가치세법 중 근로소득 실수령액 계산, 개인사업자·법인 세부담 비교, 부가가치세 "
        "신고·납부 기한, 법인 비용(손금) 인정 기준에 필요한 조문을 발췌한 문서입니다. 소득세법 "
        "본문은 별도의 data/소득세법.jsonl(법제처 국가법령정보센터 원문)을 사용하며, 이 PDF는 "
        "그 파일에 없는 법령만 보충합니다.",
        styles["body"],
    ),
    Spacer(1, 6 * mm),
]
for law, article, topic, text in TAX_SUPPLEMENT_ENTRIES:
    story.append(Paragraph(f"■ {law} | {article} ({topic})", styles["header"]))
    story.append(Paragraph(text, styles["body"]))
pdf_doc.build(story)

print(f"생성 완료: {SOCIAL_INS_PDF_PATH}")


생성 완료: data/social_insurance_source.pdf


## 2. 소득세법 원문 적재·정제

1. **삭제된 조문 제외** — `삭제여부: true`인 조문은 더 이상 유효하지 않으므로 제외합니다.
2. **중복 id 처리** — 아직 시행되지 않은(미래 시행일이 본문에 명시된) 개정 조문 버전은
   제외하고 현재 시행 중인 텍스트만 채택합니다.


In [3]:
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    raw_articles = [json.loads(line) for line in f]

FUTURE_EFFECTIVE_RE = re.compile(r"\[시행일\s*:\s*\d{4}\.")


def dedupe_by_effective_date(articles: list[dict]) -> list[dict]:
    by_id = defaultdict(list)
    for d in articles:
        by_id[d["id"]].append(d)
    result = []
    for id_, versions in by_id.items():
        if len(versions) == 1:
            result.append(versions[0])
            continue
        current = [v for v in versions if not FUTURE_EFFECTIVE_RE.search(v["본문_원본"])]
        result.append(current[0] if current else versions[0])
    return result


deduped = dedupe_by_effective_date(raw_articles)
active_articles = [d for d in deduped if not d["삭제여부"]]
# 정렬하지 않음 — dict가 삽입 순서를 보존하므로 원본 JSONL의 조문 순서(=법령 원문 순서)가
# 그대로 유지된다.

print(f"원본 조문 수: {len(raw_articles)}")
print(f"중복 제거 후: {len(deduped)}건 -> 삭제조문 제외 후 최종: {len(active_articles)}건")


원본 조문 수: 349
중복 제거 후: 326건 -> 삭제조문 제외 후 최종: 259건


## 3. 표 이미지 직접 검증 결과 반영

제47조·제55조·제59조는 원문 PDF에서 표가 이미지로만 존재해 JSONL의 `본문`에는 실제
수치가 빠져 있습니다. 아래 표는 실제 공식 PDF를 DOCX로 변환해 추출한 표 이미지를 직접
육안으로 대조해 옮긴 값입니다 — DOCX 안에는 텍스트 나열이 아니라 진짜 워드 표(Table)로
원래 있어야 할 위치에 삽입합니다.


In [4]:
TABLE_47 = [
    ("총급여액", "공제액"),
    ("500만원 이하", "총급여액의 100분의 70"),
    ("500만원 초과 1천500만원 이하", "350만원 + (500만원을 초과하는 금액의 100분의 40)"),
    ("1천500만원 초과 4천500만원 이하", "750만원 + (1천500만원을 초과하는 금액의 100분의 15)"),
    ("4천500만원 초과 1억원 이하", "1천200만원 + (4천500만원을 초과하는 금액의 100분의 5)"),
    ("1억원 초과", "1천475만원 + (1억원을 초과하는 금액의 100분의 2)"),
]

TABLE_55 = [
    ("종합소득 과세표준", "세율"),
    ("1,400만원 이하", "과세표준의 6퍼센트"),
    ("1,400만원 초과 5,000만원 이하", "84만원 + (1,400만원을 초과하는 금액의 15퍼센트)"),
    ("5,000만원 초과 8,800만원 이하", "624만원 + (5,000만원을 초과하는 금액의 24퍼센트)"),
    ("8,800만원 초과 1억5천만원 이하", "1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)"),
    ("1억5천만원 초과 3억원 이하", "3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)"),
    ("3억원 초과 5억원 이하", "9,406만원 + (3억원을 초과하는 금액의 40퍼센트)"),
    ("5억원 초과 10억원 이하", "1억7,406만원 + (5억원을 초과하는 금액의 42퍼센트)"),
    ("10억원 초과", "3억8,406만원 + (10억원을 초과하는 금액의 45퍼센트)"),
]

TABLE_59 = [
    ("근로소득에 대한 종합소득 산출세액", "공제액"),
    ("130만원 이하", "산출세액의 100분의 55"),
    ("130만원 초과", "71만5천원 + (130만원을 초과하는 금액의 100분의 30)"),
]

# id -> (표 앞 문단들, 표, 표 뒤 문단들) — 실제 조문 흐름 그대로
ARTICLE_BODY_OVERRIDES = {
    "소득세법_제47조": (
        ["①근로소득이 있는 거주자에 대해서는 해당 과세기간에 받는 총급여액에서 다음의 금액을 공제한다."],
        TABLE_47,
        [
            "다만, 공제액이 2천만원을 초과하는 경우에는 2천만원을 공제한다.",
            "② 일용근로자에 대한 공제액은 제1항에도 불구하고 1일 15만원으로 한다.",
            "③ 근로소득이 있는 거주자의 해당 과세기간의 총급여액이 제1항 또는 제2항의 공제액에 "
            "미달하는 경우에는 그 총급여액을 공제액으로 한다.",
            "④ 제1항부터 제3항까지의 규정에 따른 공제를 “근로소득공제”라 한다.",
            "⑤ 제1항의 경우에 2인 이상으로부터 근로소득을 받는 사람(일용근로자는 제외한다)에 "
            "대하여는 그 근로소득의 합계액을 총급여액으로 하여 제1항에 따라 계산한 근로소득공제액을 "
            "총급여액에서 공제한다.",
            "⑥ 삭제",
            "[전문개정 2009. 12. 31.]",
        ],
    ),
    "소득세법_제55조": (
        ["①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 "
         "계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다."],
        TABLE_55,
        [
            "② 거주자의 퇴직소득에 대한 소득세는 다음 각 호의 순서에 따라 계산한 금액(이하 "
            "“퇴직소득 산출세액”이라 한다)으로 한다.",
            "1. 해당 과세기간의 퇴직소득과세표준에 제1항의 세율을 적용하여 계산한 금액",
            "2. 제1호의 금액을 12로 나눈 금액에 근속연수를 곱한 금액",
            "3. 삭제",
            "[전문개정 2009. 12. 31.]",
        ],
    ),
    "소득세법_제59조": (
        ["①근로소득이 있는 거주자에 대해서는 그 근로소득에 대한 종합소득산출세액에서 다음의 금액을 공제한다."],
        TABLE_59,
        [
            "② 제1항에도 불구하고 공제세액이 다음 각 호의 구분에 따른 금액을 초과하는 경우에 그 "
            "초과하는 금액은 없는 것으로 한다.",
            "1. 총급여액이 3천 300만원 이하인 경우: 74만원",
            "2. 총급여액이 3천 300만원 초과 7천만원 이하인 경우: 74만원 - [(총급여액 - 3천 300만원) "
            "× 8/1000]. 다만, 위 금액이 66만원보다 적은 경우에는 66만원으로 한다.",
            "3. 총급여액이 7천만원 초과 1억2천만원 이하인 경우: 66만원 - [(총급여액 - 7천만원) × "
            "1/2]. 다만, 위 금액이 50만원보다 적은 경우에는 50만원으로 한다.",
            "4. 총급여액이 1억2천만원을 초과하는 경우: 50만원 - [(총급여액 - 1억2천만원) × 1/2]. "
            "다만, 위 금액이 20만원보다 적은 경우에는 20만원으로 한다.",
            "③ 일용근로자의 근로소득에 대해서 제134조제3항에 따른 원천징수를 하는 경우에는 해당 "
            "근로소득에 대한 산출세액의 100분의 55에 해당하는 금액을 그 산출세액에서 공제한다.",
            "[전문개정 2012. 1. 1.]",
        ],
    ),
}

print(f"표 원위치 삽입 대상 조문: {len(ARTICLE_BODY_OVERRIDES)}건")


표 원위치 삽입 대상 조문: 3건


## 4. DOCX 조립

소득세법 조문(표 3개는 실제 워드 표로 삽입) + 4대보험 보충 조문을 `data/소득세법_통합.docx`
하나로 저장합니다. 편집자 주석이나 색상 표시는 넣지 않습니다 — RAG 파이프라인이 곧바로
읽어들일 원본이므로, 실제 법령 문서를 그대로 옮긴 형태를 유지합니다.


In [5]:
from docx import Document
from docx.shared import Pt
from pypdf import PdfReader


def parse_law_pdf(text: str) -> list[dict]:
    blocks = re.split(r"(?m)^■\s*", text)
    header_re = re.compile(r"(?P<law>.+?)\s*\|\s*(?P<article>.+?)\s*\((?P<topic>[^)]*)\)\s*")
    docs = []
    for block in blocks[1:]:
        header_line, _, rest = block.partition("\n")
        m = header_re.match(header_line.strip())
        if not m:
            continue
        docs.append({
            "law": m.group("law").strip(),
            "article": m.group("article").strip(),
            "topic": m.group("topic").strip(),
            "text": " ".join(rest.split()),
        })
    return docs


reader = PdfReader(SOCIAL_INS_PDF_PATH)
pdf_text = "\n".join(page.extract_text() for page in reader.pages)
tax_supplement_articles = parse_law_pdf(pdf_text)


def add_table(doc, rows):
    table = doc.add_table(rows=len(rows), cols=2)
    table.style = "Light Grid Accent 1"
    for r, (left, right) in enumerate(rows):
        table.cell(r, 0).text = left
        table.cell(r, 1).text = right
        if r == 0:
            for cell in table.rows[0].cells:
                for p in cell.paragraphs:
                    for run in p.runs:
                        run.bold = True
    doc.add_paragraph()


docx_doc = Document()
style = docx_doc.styles["Normal"]
style.font.name = "맑은 고딕"
style.font.size = Pt(10.5)

meta = raw_articles[0]
docx_doc.add_paragraph("법제처                                                                    국가법령정보센터")
docx_doc.add_heading("소득세법", level=0)
docx_doc.add_paragraph(f"[시행 {meta['시행일']}] [법률 {meta['법령번호']}, {meta['공포일']}, 일부개정]")

current_chapter = None
for a in active_articles:
    if a["장"] != current_chapter:
        current_chapter = a["장"]
        if current_chapter:
            docx_doc.add_paragraph(current_chapter)

    docx_doc.add_heading(f"{a['조번호']}({a['조제목']})", level=3)

    if a["id"] in ARTICLE_BODY_OVERRIDES:
        before, table_rows, after = ARTICLE_BODY_OVERRIDES[a["id"]]
        for line in before:
            docx_doc.add_paragraph(line)
        add_table(docx_doc, table_rows)
        for line in after:
            docx_doc.add_paragraph(line)
    else:
        for line in a["본문"].split("\n"):
            if line.strip():
                docx_doc.add_paragraph(line.strip())

docx_doc.add_heading("4대보험·지방소득세·법인세율 관련 법령 (보충)", level=1)
for d in tax_supplement_articles:
    docx_doc.add_heading(f"{d['law']} {d['article']}({d['topic']})", level=3)
    docx_doc.add_paragraph(d["text"])

docx_doc.save(OUT_PATH)
print(f"생성 완료: {OUT_PATH}")
print(f"  - 소득세법 조문: {len(active_articles)}건 (표 3개는 실제 Word 표로 원위치에 삽입)")
print(f"  - 세법 보충 조문(4대보험+법인세율): {len(tax_supplement_articles)}건")


생성 완료: data/소득세법_통합.docx
  - 소득세법 조문: 259건 (표 3개는 실제 Word 표로 원위치에 삽입)
  - 세법 보충 조문(4대보험+법인세율): 9건


## 완료

`data/소득세법_통합.docx`가 준비됐습니다. 이제 `income_tax_rag_demo.ipynb`를 열어
실행하면 이 DOCX를 직접 파싱해 RAG 파이프라인이 시작됩니다.
